# P1: Framing a Data Problem

**DSA 405 · Fall 2026 · Project Milestone 1**

Name: Sharmila  |  Unity ID: sshirol

## 1. The Question

Do certain hard drive manufacturers and models fail earlier under real datacenter workloads, and can drives heading toward failure be flagged in advance from their operating data (SMART attributes)?

This question matters to anyone who runs storage at scale, such as a datacenter operator, a backup provider, or an IT team deciding what hardware to buy and when to replace it. A drive failure is costly twice, once in the replacement itself and once in the risk to the data stored on it. If failure rates differ meaningfully by model, or if a small set of SMART attributes reliably rises before a failure, that information changes both purchasing decisions and replacement timing. An answer would let an operator retire the weakest models sooner and remove at-risk drives before they fail rather than after.

The question is answerable because the table that would settle it can be described: one row per drive, with the manufacturer, the model, the published specifications (capacity, rotational speed, warranty), the number of days in service, a failure flag, and the SMART readings. I expect the exact wording to narrow once the data is in hand, and I will document that shift in P4. For P1, the goal is to establish that a real, answerable question exists.

## 2. Sources & Access Evidence

### Source 1 (download): Backblaze Drive Stats

| | |
|---|---|
| **Publisher** | Backblaze, Inc., hosted on its own Backblaze B2 storage. |
| **URL** | https://www.backblaze.com/cloud-storage/resources/hard-drive-test-data |
| **Coverage & time span** | Daily snapshots of every drive in Backblaze's datacenters, 2013 through Q1 2026. Each record contains the date, serial number, model, capacity, a failure flag, and SMART attributes. |
| **Approximate size** | Q1 2026 alone tracks 341,263 drives across roughly 30.2 million drive-days (about 1.3 GB compressed). I will work with one quarter and a reduced set of columns. |
| **Access method** | Download of zipped CSV files. This is my download source. |

### Source 2 (off the web): Wikipedia drive specification tables

| | |
|---|---|
| **Publisher** | Wikipedia / Wikimedia Foundation. |
| **URL** | Manufacturer product-line articles, for example https://en.wikipedia.org/wiki/Western_Digital_Red and https://en.wikipedia.org/wiki/Seagate_Barracuda. I will pin the exact articles to the models that appear in the Backblaze quarter I use. |
| **Coverage & time span** | Currently documented specifications per model line: capacity, rotational speed, cache, interface, and form factor. |
| **Approximate size** | Tens of model rows per article, on the order of 100 to 250 models across the articles used. |
| **Access method** | read_html over the article HTML. This is my off-the-web source. |

**How the two sources combine.** Backblaze provides the failure and operating data, and Wikipedia provides the design specifications. The two join one-to-many on the drive model string, where one specification row maps to the many drives of that model in the Backblaze data. Once joined, I can compute a failure rate for each model and each manufacturer and test whether specifications such as capacity or warranty length relate to failure. Neither source answers the question on its own, which is why the project requires both.

### Evidence each source is reachable

The cell below loads the Backblaze schema, confirms the data file responds, and confirms the Wikipedia article responds and returns tables.

In [3]:
import io, requests, pandas as pd

# Source 1 (Backblaze): load the schema file, confirm the data file exists
SCHEMA = "https://f001.backblazeb2.com/file/Backblaze-Hard-Drive-Data/Drive_Stats_Schema_Current.csv"
print("Backblaze schema shape:", pd.read_csv(SCHEMA).shape)
ZIP = "https://f001.backblazeb2.com/file/Backblaze-Hard-Drive-Data/data_Q1_2026.zip"
print("Backblaze data file status:", requests.head(ZIP, timeout=30, allow_redirects=True).status_code)

# Source 2 (Wikipedia): confirm the page responds and read_html finds tables
UA = {"User-Agent": "DSA405-student-lab/1.0 (NC State class exercise)"}
SPEC_URL = "https://en.wikipedia.org/wiki/Seagate_Barracuda"
r = requests.get(SPEC_URL, headers=UA, timeout=30)
try:
    n = len(pd.read_html(io.StringIO(r.text)))
except ValueError:
    n = 0
print("Wikipedia page status:", r.status_code, "| tables found:", n)

Backblaze schema shape: (197, 1)
Backblaze data file status: 200
Wikipedia page status: 200 | tables found: 9


**Likeliest failure point.** The least certain source is Wikipedia, because its model strings will not all match the Backblaze model strings exactly, so some rows will not join. If per-model matching proves too lossy, I will aggregate the specifications to the manufacturer and capacity-class level and join on that instead. That fallback still supports the failure-by-manufacturer part of the question, and the SMART early-warning part is unaffected because it relies only on the Backblaze data.

## 3. Constraints & Guardrails

**Source 1, Backblaze Drive Stats.** Backblaze publishes the data under stated terms of use. The relevant conditions are:

> "Cite Backblaze as the source if you use the data."
>
> "You are solely responsible for how you use the data."
>
> "You can not sell the data itself (you may sell derivative works based on the data)."

Use of the data is free of charge. My use is non-commercial coursework and I will cite Backblaze as the source, so the planned use is permitted. No login or payment is involved.

**Source 2, Wikipedia.** Wikipedia's robots.txt is published at https://en.wikipedia.org/robots.txt. For a general scraper it disallows the query endpoints under /w/ but does not disallow article pages under /wiki/, so reading a /wiki/ article is permitted. The relevant lines are:

> User-agent: *
>
> Disallow: /w/
>
> Disallow: /wiki/Special:

Wikipedia article content is licensed under Creative Commons Attribution-ShareAlike (CC BY-SA), which permits reuse with attribution, and I will attribute Wikipedia as the source.

**Course guardrails.**

- **Scrape only permitted sites.** The off-the-web source is Wikipedia, whose robots.txt permits reading article paths. The Backblaze source is an open, licensed download, not a scrape.
- **Never data behind a login or paywall.** Both sources are public and free.
- **Never personal or identifiable information.** The data describes drives, not people. Serial numbers identify hardware, not individuals.
- **Always rate-limit, identify honestly, and cache.** My requests use an honest User-Agent identifying the work as an NC State class exercise, run with a delay between calls, and cache responses locally so pages are not requested more than once.

## 4. Tier Declaration

**Tier 1 (Solid).** The project uses two sources, one obtained off the web with read_html, joined one-to-many on the drive model. This fits me and these sources because the combination and the join are achievable within the course timeframe without pagination, session handling, or extraction from unstructured text. I may raise the tier later, which is allowed up to P3, if the specification data turns out to require a multi-page scrape.

## Self-Scored Rubric

Scale: 3 = strong, evidenced, specific; 2 = present but generic or asserted; 1 = weak or partial; 0 = missing. Weighted = score x weight.

| Criterion | Wt | Score | Weighted | Justification |
|---|---|---|---|---|
| Question & motivation | x1 | 3 | 3 | One-sentence question with a verb, answerable by a table I can describe. The motivation names real stakeholders and what an answer changes. |
| Sources & access evidence | x2 | 3 | 6 | Two sources, one download and one off the web, that must be combined and join one-to-many on model. Publisher, URL, coverage, size, and access method are given for each, reachability is shown by the recorded output, and the failure point is named with a fallback. |
| Applicable constraints & guardrails | x2 | 3 | 6 | The Backblaze terms and the Wikipedia robots.txt lines are quoted, and all four course guardrails are addressed by name. |
| Tier declaration & fit | x1 | 3 | 3 | Tier 1 is declared with one sentence on why it fits the project and the sources. |
| **Total** | | | **18 / 18** | |

## P1 Checklist

- [x] One sentence stating the question, answerable with the named data
- [x] Two or more sources, each with publisher, URL, coverage, and size
- [x] At least one source comes off the web (scrape or API)
- [x] Evidence each source is reachable, pasted in
- [x] The constraint that applies to each source, quoted
- [x] Every course guardrail addressed by name
- [x] Tier declared, with one sentence on why
- [x] Self-scored rubric attached